# **BizFlow360 XGBoost Notebook**

* **By:** Edusei Mikel
* **Date** 6th August, 2026

**Imports and Data Loading**

In [3]:
import pandas as pd
import numpy as np
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import xgboost as xgb

# Load the synthetic data
df = pd.read_csv('../data/synthetic_msme_data.csv')

# Encode categorical variables
le_county = LabelEncoder()
le_sector = LabelEncoder()
df['county_encoded'] = le_county.fit_transform(df['county'])
df['sector_encoded'] = le_sector.fit_transform(df['sector'])

# Define Features (X) and Target (y)
features = [
    'business_age_months', 'employees', 
    'monthly_revenue_kes', 'monthly_expenses_kes', 
    'total_assets_kes', 'total_liabilities_kes', 
    'loan_amount_kes', 'mpesa_volume_kes', 
    'county_encoded', 'sector_encoded'
]

X = df[features]
y = df['distress_label']

# Split and Scale (Same split/scaler for fair comparison)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("✅ Data loaded, preprocessed, and split successfully.")

✅ Data loaded, preprocessed, and split successfully.


**Training the XGBoost Model**

In [5]:
print("Training XGBoost Model...")

# Initialize and train the model
# use_label_encoder=False suppresses a common warning in newer XGBoost versions
# eval_metric='logloss' is the standard for binary classification
xgb_model = xgb.XGBClassifier(
    n_estimators=100, 
    learning_rate=0.1, 
    max_depth=5, 
    use_label_encoder=False, 
    eval_metric='logloss', 
    random_state=42
)

xgb_model.fit(X_train_scaled, y_train)

print("✅ XGBoost model trained successfully!")

Training XGBoost Model...


/home/mikel/BizFlow360/venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [16:00:32] WARNING: /__w/xgboost/xgboost/src/learner.cc:794: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


✅ XGBoost model trained successfully!


**Model Evaluation**

In [6]:
# Make predictions
y_pred = xgb_model.predict(X_test_scaled)
y_prob = xgb_model.predict_proba(X_test_scaled)[:, 1]

# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_prob)

print("="*40)
print(" XGBOOST MODEL METRICS")
print("="*40)
print(f"Accuracy:  {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"F1-Score:  {f1:.4f}")
print(f"ROC-AUC:   {roc_auc:.4f}")
print("="*40)

 XGBOOST MODEL METRICS
Accuracy:  0.7380
Precision: 0.7380
Recall:    0.7380
F1-Score:  0.7380
ROC-AUC:   0.8365


**Saving the Model**

In [7]:
# Ensure trained models directory exists
os.makedirs('../models/trained', exist_ok=True)

# Save the model
joblib.dump(xgb_model, '../models/trained/xgboost.joblib')

print("✅ XGBoost model saved to edusei_ml/models/trained/xgboost.joblib")

✅ XGBoost model saved to edusei_ml/models/trained/xgboost.joblib
